# KD/Pruning/Quantization Experiments for Bangla Cyberbullying Detection

This notebook contains 8 experiment scenarios (23 total runs) to compare different model compression techniques.

## Models
| Type | Model Path | Description |
|------|------------|-------------|
| Teacher | `Saif-Siddique/bangla-cyberbully-xlm-roberta-base` | XLM-RoBERTa based |
| Finetuned Student | `Saif-Siddique/bangla-cyberbully-neuropark-sahajBERT` | SahajBERT finetuned |
| Raw Student | `neuropark/sahajBERT` | SahajBERT (not finetuned) |

## Scenarios Overview
| Scenario | Description | Runs |
|----------|-------------|------|
| 1 | Baseline (Teacher + Finetuned Student) | 2 |
| 2 | KD Only | 1 |
| 3 | Prune Only | 2 |
| 4 | Quant Only | 2 |
| 5 | Prune + Quant | 2 |
| 6 | Pruning Methods (magnitude/wanda/gradual) | 3 |
| 7 | Quant Methods (fp16/int8/int4) | 3 |
| 8 | KD Hyperparameter Tuning | 8 |
| **Total** | | **23** |

---
## Setup

In [ ]:
# Install dependencies
!pip install -q transformers datasets torch scikit-learn pandas numpy accelerate bitsandbytes iterstrat

In [ ]:
# Clone the framework repository
!git clone https://github.com/Saif-Siddique/kd_pruning_quantization_framework_for_nlp.git
%cd kd_pruning_quantization_framework_for_nlp

In [ ]:
# Verify setup
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Define constants
DATASET_PATH = "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv"
AUTHOR_NAME = "Saif-Siddique"

# Models
TEACHER = "Saif-Siddique/bangla-cyberbully-xlm-roberta-base"
FINETUNED_STUDENT = "Saif-Siddique/bangla-cyberbully-neuropark-sahajBERT"
RAW_STUDENT = "neuropark/sahajBERT"

---
## Scenario 1: Baseline Performance Test (2 runs)

Evaluate Teacher and Finetuned Student models without any compression.

**Purpose:** Establish baseline metrics for comparison.

In [ ]:
# Run 1.1: Teacher Baseline
!python main.py \
    --dataset_path "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline baseline \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-xlm-roberta-base" \
    --use_original_folds \
    --eval_fold 3 \
    --output_dir ./results/scenario1/teacher_baseline

In [ ]:
# Run 1.2: Finetuned Student Baseline
!python main.py \
    --dataset_path "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline baseline \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-neuropark-sahajBERT" \
    --use_original_folds \
    --eval_fold 3 \
    --output_dir ./results/scenario1/finetuned_student_baseline

---
## Scenario 2: KD Only (1 run)

Apply Knowledge Distillation from Teacher to Raw Student.

**Purpose:** Compare KD-trained student with directly finetuned student.

**Default KD Parameters:**
- Alpha: 0.7 (70% soft labels, 30% hard labels)
- Temperature: 4.0
- Method: logit
- Epochs: 15

In [ ]:
# Run 2.1: KD from Teacher to Raw Student
!python main.py \
    --dataset_path "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline kd_only \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-xlm-roberta-base" \
    --student_path "neuropark/sahajBERT" \
    --use_original_folds \
    --eval_fold 3 \
    --output_dir ./results/scenario2/kd_only

---
## Scenario 3: Prune Only (2 runs)

Apply pruning to both Finetuned Student and Distilled Student, then fine-tune.

**Purpose:** Compare pruning effectiveness on finetuned vs distilled models.

**Default Pruning Parameters:**
- Method: magnitude
- Sparsity: 50%
- Fine-tune after: Yes (3 epochs)

In [ ]:
# Run 3.1: Prune Finetuned Student
!python main.py \
    --dataset_path "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline prune_only \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-neuropark-sahajBERT" \
    --prune_method magnitude \
    --prune_sparsity 0.5 \
    --fine_tune_after_prune \
    --use_original_folds \
    --eval_fold 3 \
    --output_dir ./results/scenario3/prune_finetuned

In [ ]:
# Run 3.2: Prune Distilled Student (KD -> Prune)
!python main.py \
    --dataset_path "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline kd_prune \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-xlm-roberta-base" \
    --student_path "neuropark/sahajBERT" \
    --prune_method magnitude \
    --prune_sparsity 0.5 \
    --fine_tune_after_prune \
    --use_original_folds \
    --eval_fold 3 \
    --output_dir ./results/scenario3/prune_distilled

---
## Scenario 4: Quant Only (2 runs)

Apply quantization to both Finetuned Student and Distilled Student.

**Purpose:** Compare quantization effectiveness on finetuned vs distilled models.

**Default Quantization Parameters:**
- Method: dynamic
- Data type: int8

In [ ]:
# Run 4.1: Quantize Finetuned Student
!python main.py \
    --dataset_path "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline quant_only \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-neuropark-sahajBERT" \
    --quant_method dynamic \
    --use_original_folds \
    --eval_fold 3 \
    --output_dir ./results/scenario4/quant_finetuned

In [ ]:
# Run 4.2: Quantize Distilled Student (KD -> Quant)
!python main.py \
    --dataset_path "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline kd_quant \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-xlm-roberta-base" \
    --student_path "neuropark/sahajBERT" \
    --quant_method dynamic \
    --use_original_folds \
    --eval_fold 3 \
    --output_dir ./results/scenario4/quant_distilled

---
## Scenario 5: Prune + Quant (2 runs)

Apply both pruning and quantization.

**Purpose:** Compare full compression pipeline on finetuned vs distilled models.

In [ ]:
# Run 5.1: Prune + Quant Finetuned Student
!python main.py \
    --dataset_path "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline prune_quant \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-neuropark-sahajBERT" \
    --prune_method magnitude \
    --prune_sparsity 0.5 \
    --quant_method dynamic \
    --use_original_folds \
    --eval_fold 3 \
    --output_dir ./results/scenario5/prune_quant_finetuned

In [ ]:
# Run 5.2: KD + Prune + Quant (Full Pipeline)
!python main.py \
    --dataset_path "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline kd_prune_quant \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-xlm-roberta-base" \
    --student_path "neuropark/sahajBERT" \
    --prune_method magnitude \
    --prune_sparsity 0.5 \
    --quant_method dynamic \
    --use_original_folds \
    --eval_fold 3 \
    --output_dir ./results/scenario5/kd_prune_quant

---
## Scenario 6: Pruning Methods Comparison (3 runs)

Compare magnitude, wanda, and gradual pruning on Distilled Student.

**Purpose:** Find the best pruning method for this task.

**Methods:**
- **Magnitude:** Removes weights with smallest absolute values
- **Wanda:** Pruning by Weights and Activations (data-aware)
- **Gradual:** Gradually increases sparsity during training

In [ ]:
# Run 6.1: Magnitude Pruning
!python main.py \
    --dataset_path "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline kd_prune \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-xlm-roberta-base" \
    --student_path "neuropark/sahajBERT" \
    --prune_method magnitude \
    --prune_sparsity 0.5 \
    --use_original_folds \
    --eval_fold 3 \
    --output_dir ./results/scenario6/magnitude

In [ ]:
# Run 6.2: Wanda Pruning
!python main.py \
    --dataset_path "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline kd_prune \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-xlm-roberta-base" \
    --student_path "neuropark/sahajBERT" \
    --prune_method wanda \
    --prune_sparsity 0.5 \
    --use_original_folds \
    --eval_fold 3 \
    --output_dir ./results/scenario6/wanda

In [ ]:
# Run 6.3: Gradual Pruning
!python main.py \
    --dataset_path "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline kd_prune \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-xlm-roberta-base" \
    --student_path "neuropark/sahajBERT" \
    --prune_method gradual \
    --prune_sparsity 0.5 \
    --prune_schedule cubic \
    --use_original_folds \
    --eval_fold 3 \
    --output_dir ./results/scenario6/gradual

---
## Scenario 7: Quantization Methods Comparison (3 runs)

Compare fp16, int8 (dynamic), and int4 quantization on Distilled Student.

**Purpose:** Find the best quantization method balancing size reduction vs accuracy.

**Methods:**
- **FP16:** Half-precision floating point (requires GPU)
- **INT8 Dynamic:** 8-bit integer quantization (CPU compatible)
- **INT4:** 4-bit quantization (requires bitsandbytes)

In [ ]:
# Run 7.1: FP16 Quantization
!python main.py \
    --dataset_path "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline kd_quant \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-xlm-roberta-base" \
    --student_path "neuropark/sahajBERT" \
    --quant_method fp16 \
    --use_original_folds \
    --eval_fold 3 \
    --output_dir ./results/scenario7/fp16

In [ ]:
# Run 7.2: INT8 Dynamic Quantization
!python main.py \
    --dataset_path "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline kd_quant \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-xlm-roberta-base" \
    --student_path "neuropark/sahajBERT" \
    --quant_method dynamic \
    --use_original_folds \
    --eval_fold 3 \
    --output_dir ./results/scenario7/int8

In [ ]:
# Run 7.3: INT4 Quantization
!python main.py \
    --dataset_path "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline kd_quant \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-xlm-roberta-base" \
    --student_path "neuropark/sahajBERT" \
    --quant_method int4 \
    --use_original_folds \
    --eval_fold 3 \
    --output_dir ./results/scenario7/int4

---
## Scenario 8: KD Hyperparameter Tuning (8 runs)

Grid search over alpha, temperature, and learning rate.

**Purpose:** Find optimal KD hyperparameters.

| Run | Alpha | Temperature | Learning Rate |
|-----|-------|-------------|---------------|
| 8.1 | 0.5 | 2.0 | 1e-5 |
| 8.2 | 0.5 | 2.0 | 5e-5 |
| 8.3 | 0.5 | 6.0 | 1e-5 |
| 8.4 | 0.5 | 6.0 | 5e-5 |
| 8.5 | 0.9 | 2.0 | 1e-5 |
| 8.6 | 0.9 | 2.0 | 5e-5 |
| 8.7 | 0.9 | 6.0 | 1e-5 |
| 8.8 | 0.9 | 6.0 | 5e-5 |

In [ ]:
# Run 8.1: alpha=0.5, T=2.0, lr=1e-5
!python main.py \
    --dataset_path "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline kd_only \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-xlm-roberta-base" \
    --student_path "neuropark/sahajBERT" \
    --kd_alpha 0.5 \
    --kd_temperature 2.0 \
    --lr 1e-5 \
    --use_original_folds \
    --eval_fold 3 \
    --output_dir ./results/scenario8/hp_a05_t2_lr1e5

In [ ]:
# Run 8.2: alpha=0.5, T=2.0, lr=5e-5
!python main.py \
    --dataset_path "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline kd_only \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-xlm-roberta-base" \
    --student_path "neuropark/sahajBERT" \
    --kd_alpha 0.5 \
    --kd_temperature 2.0 \
    --lr 5e-5 \
    --use_original_folds \
    --eval_fold 3 \
    --output_dir ./results/scenario8/hp_a05_t2_lr5e5

In [ ]:
# Run 8.3: alpha=0.5, T=6.0, lr=1e-5
!python main.py \
    --dataset_path "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline kd_only \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-xlm-roberta-base" \
    --student_path "neuropark/sahajBERT" \
    --kd_alpha 0.5 \
    --kd_temperature 6.0 \
    --lr 1e-5 \
    --use_original_folds \
    --eval_fold 3 \
    --output_dir ./results/scenario8/hp_a05_t6_lr1e5

In [ ]:
# Run 8.4: alpha=0.5, T=6.0, lr=5e-5
!python main.py \
    --dataset_path "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline kd_only \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-xlm-roberta-base" \
    --student_path "neuropark/sahajBERT" \
    --kd_alpha 0.5 \
    --kd_temperature 6.0 \
    --lr 5e-5 \
    --use_original_folds \
    --eval_fold 3 \
    --output_dir ./results/scenario8/hp_a05_t6_lr5e5

In [ ]:
# Run 8.5: alpha=0.9, T=2.0, lr=1e-5
!python main.py \
    --dataset_path "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline kd_only \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-xlm-roberta-base" \
    --student_path "neuropark/sahajBERT" \
    --kd_alpha 0.9 \
    --kd_temperature 2.0 \
    --lr 1e-5 \
    --use_original_folds \
    --eval_fold 3 \
    --output_dir ./results/scenario8/hp_a09_t2_lr1e5

In [ ]:
# Run 8.6: alpha=0.9, T=2.0, lr=5e-5
!python main.py \
    --dataset_path "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline kd_only \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-xlm-roberta-base" \
    --student_path "neuropark/sahajBERT" \
    --kd_alpha 0.9 \
    --kd_temperature 2.0 \
    --lr 5e-5 \
    --use_original_folds \
    --eval_fold 3 \
    --output_dir ./results/scenario8/hp_a09_t2_lr5e5

In [ ]:
# Run 8.7: alpha=0.9, T=6.0, lr=1e-5
!python main.py \
    --dataset_path "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline kd_only \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-xlm-roberta-base" \
    --student_path "neuropark/sahajBERT" \
    --kd_alpha 0.9 \
    --kd_temperature 6.0 \
    --lr 1e-5 \
    --use_original_folds \
    --eval_fold 3 \
    --output_dir ./results/scenario8/hp_a09_t6_lr1e5

In [ ]:
# Run 8.8: alpha=0.9, T=6.0, lr=5e-5
!python main.py \
    --dataset_path "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline kd_only \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-xlm-roberta-base" \
    --student_path "neuropark/sahajBERT" \
    --kd_alpha 0.9 \
    --kd_temperature 6.0 \
    --lr 5e-5 \
    --use_original_folds \
    --eval_fold 3 \
    --output_dir ./results/scenario8/hp_a09_t6_lr5e5

---
## Results Aggregation

Combine all CSV results into a summary table for analysis.

In [ ]:
import pandas as pd
import os
from glob import glob

def collect_results(base_dir='./results'):
    """Collect all results CSV files into a single DataFrame."""
    all_results = []
    
    # Find all results_final.csv or results_*_fold*.csv files
    for csv_path in glob(f"{base_dir}/**/results_*.csv", recursive=True):
        try:
            df = pd.read_csv(csv_path)
            # Extract scenario and experiment name from path
            parts = csv_path.replace('\\', '/').split('/')
            scenario = next((p for p in parts if p.startswith('scenario')), 'unknown')
            experiment = parts[-2] if len(parts) > 2 else 'unknown'
            
            df['scenario'] = scenario
            df['experiment'] = experiment
            df['source_file'] = csv_path
            all_results.append(df)
        except Exception as e:
            print(f"Error reading {csv_path}: {e}")
    
    if all_results:
        return pd.concat(all_results, ignore_index=True)
    return pd.DataFrame()

# Collect all results
results_df = collect_results()
print(f"Collected {len(results_df)} result rows from all scenarios")

In [ ]:
# Display summary table
if not results_df.empty:
    # Select key columns for summary
    summary_cols = ['scenario', 'experiment', 'stage', 'size_mb', 'params_M', 
                    'sparsity_%', 'f1_macro', 'f1_weighted', 'acc_exact',
                    'latency_ms', 'compression', 'speedup']
    
    available_cols = [c for c in summary_cols if c in results_df.columns]
    summary_df = results_df[available_cols].copy()
    
    # Sort by scenario and experiment
    summary_df = summary_df.sort_values(['scenario', 'experiment', 'stage'])
    
    print("\n" + "="*100)
    print("EXPERIMENT RESULTS SUMMARY")
    print("="*100)
    display(summary_df)
else:
    print("No results found. Make sure experiments have been run.")

In [ ]:
# Save aggregated results
if not results_df.empty:
    output_path = './results/all_experiments_summary.csv'
    results_df.to_csv(output_path, index=False)
    print(f"Aggregated results saved to: {output_path}")
    
    # Also create a pivot table for easy comparison
    if 'f1_macro' in results_df.columns:
        pivot_df = results_df.pivot_table(
            index=['scenario', 'experiment'],
            columns='stage',
            values=['f1_macro', 'size_mb', 'latency_ms'],
            aggfunc='first'
        )
        pivot_df.to_csv('./results/experiments_pivot.csv')
        print("Pivot table saved to: ./results/experiments_pivot.csv")

In [ ]:
# Per-label F1 comparison (if available)
if not results_df.empty:
    label_cols = ['f1_bully', 'f1_sexual', 'f1_religious', 'f1_threat', 'f1_spam']
    available_label_cols = [c for c in label_cols if c in results_df.columns]
    
    if available_label_cols:
        print("\n" + "="*100)
        print("PER-LABEL F1 SCORES")
        print("="*100)
        
        label_summary = results_df[['scenario', 'experiment', 'stage'] + available_label_cols].copy()
        display(label_summary)

---
## Analysis Helper Functions

In [ ]:
def compare_scenarios(results_df, metric='f1_macro'):
    """Compare final stage results across all scenarios."""
    if results_df.empty:
        print("No results to compare")
        return
    
    # Get final stage for each experiment
    final_results = results_df.groupby(['scenario', 'experiment']).last().reset_index()
    
    print(f"\nComparison by {metric}:")
    print("-" * 60)
    
    for scenario in sorted(final_results['scenario'].unique()):
        scenario_data = final_results[final_results['scenario'] == scenario]
        print(f"\n{scenario.upper()}:")
        for _, row in scenario_data.iterrows():
            val = row.get(metric, 'N/A')
            if isinstance(val, float):
                print(f"  {row['experiment']}: {val:.4f}")
            else:
                print(f"  {row['experiment']}: {val}")

# Example usage (uncomment after running experiments):
# compare_scenarios(results_df, 'f1_macro')

In [ ]:
def find_best_hyperparams(results_df, scenario='scenario8', metric='f1_macro'):
    """Find best hyperparameters from scenario 8."""
    if results_df.empty:
        print("No results available")
        return
    
    hp_results = results_df[results_df['scenario'] == scenario].copy()
    if hp_results.empty:
        print(f"No results for {scenario}")
        return
    
    # Get final stage results
    final_hp = hp_results.groupby('experiment').last().reset_index()
    
    # Sort by metric
    final_hp = final_hp.sort_values(metric, ascending=False)
    
    print(f"\nBest hyperparameters by {metric}:")
    print("-" * 60)
    for i, (_, row) in enumerate(final_hp.head(5).iterrows(), 1):
        print(f"{i}. {row['experiment']}: {row[metric]:.4f}")
    
    return final_hp.iloc[0]['experiment']

# Example usage (uncomment after running experiments):
# best_hp = find_best_hyperparams(results_df)